# Main Wave Diffusion Inference Script
This is the main script used to create samples using the direct wave diffusion

## Imports

### Required Imports

In [1]:
try: import librosa
except: 
    !pip install librosa

In [2]:
#Set Dir 
import sys, os
sys.path.append(os.path.abspath('..'))

# Torch
import torch
from torch import nn, Tensor
import torch.optim as optim

# Utils
import numpy as np
import logging
import matplotlib.pyplot as plt


# Base Scripts
from Libraries.Utils import *
from Libraries.U_Net import *
from Libraries.Diffusion import *

### Imports for Visualizing

In [3]:
from torchviz import make_dot
from torch.utils.tensorboard import SummaryWriter
import tensorboard
from tensorboard import notebook
from torchinfo import summary

## Setup

### Logging

In [4]:
logging_level: int = logging.INFO #LIGHT_DEBUG
logging.basicConfig(level=logging_level, format='%(asctime)s - %(levelname)s - %(message)s')
logger: logging.Logger = logging.getLogger(__name__)

### Initial Setup

In [5]:
remote_kernel: bool = True
device: str = "cuda" if torch.cuda.is_available() else "cpu"
test_data_name: str = "../Data/unseen_test_data.npy"
model_name: str = "WaveDiffusion_v6"
full_model_path: str = OS().path_to_remote_path("../Models/{}.pth".format(model_name), remote_kernel)

### Hyperparameters

In [6]:
batch_size: int = 16
seq_length: int = 2**17
train_v_obj: bool = True

### Data loading
Optional, needed for seeded inference

In [7]:
md = ModelData()
md.load_data_from_path(data_path=OS().path_to_remote_path(test_data_name, remote_kernel))
md.create_validation_split()
test_dataset, _ = md.create_datasets()
test_dataloader, _ = md.create_dataloaders(batch_size, num_workers=1)
logger.info(f"Created test dataset with length {len(md.train_dataset)}")

## Model Setup

In [8]:
u_net = UNet(in_channels=1, n_layers=4, base_channels=48, embed_dim=128, timesteps=1000, v_obj_sampler=True, kernel_size=11).to(device)

### Load Model

In [9]:
if os.path.exists(full_model_path):
    model = torch.load(full_model_path, map_location=device)
    u_net.load_state_dict(model["u_net"])
    logger.info(f"Model {model_name} loaded with {TrainingUtils().count_params(u_net)} Parameters")
else: 
    logger.info(f"Model {model_name} could not be loaded")

## Diffusion Class Setup

In [10]:
torch.backends.cudnn.benchmark = True
diffusion = Diffusion(noise_steps=1000, schedule="linear", inp_shape=[batch_size, 1, seq_length], device=device)

## Inference

### Normal Inference

In [13]:
n_steps = 200
if not train_v_obj:
    out = diffusion.bwd_diffusion_ddim(u_net, [batch_size // 2, 1, seq_length], n_steps=n_steps, eta=0)
else:
    out = diffusion.bwd_diffusion_v_obj(u_net, [batch_size // 2, 1, seq_length], n_steps=n_steps)

out = np.clip(out, -1, 1)
for i in range(batch_size // 2):
    ad = AudioData(out[i][0])
    ad.save_audio_file(f"Results/{model_name}_sample_{i + 9:02d}.wav")
    TrainingUtils().visualize_audio_and_spect(out[i][0])


### Inference with seed

In [16]:
idx = 0
n_steps = 100
iter_data = iter(test_dataloader)
seed = next(iter_data)[0].to(device).unsqueeze(1)
seed = seed[..., :seq_length]
if not train_v_obj:
    out = diffusion.bwd_diffusion_ddim(u_net, [batch_size // 2, 1, seq_length], n_steps=n_steps, eta=0, seed=seed[:batch_size // 2], seed_fwd_steps = n_steps - 10)
else:
    out = diffusion.bwd_diffusion_v_obj(u_net, [batch_size // 2, 1, seq_length], n_steps=n_steps, seed=seed[:batch_size // 2], seed_fwd_steps = n_steps - 10)

logger.info("Sample: ")
out = np.clip(out, -1, 1)
for i in range(batch_size // 2):
    ad = AudioData(out[i][0])
    ad.save_audio_file(f"Results/{model_name}_seeded_sample_{i+1:02d}.wav")
    TrainingUtils().visualize_audio_and_spect(out[i][0])

logger.info("Seed: ")
TrainingUtils().visualize_audio_and_spect(seed[idx, 0].cpu().numpy())

## Save Model Architecture

### Torchviz

In [ ]:
x, t = torch.randn(1, 1, seq_length), torch.randn(1)
y = u_net(x, t)
make_dot(y, params=dict(u_net.named_parameters())).render(OS().path_to_remote_path(f"../Results/{model_name}", remote_kernel), format="png")

### Tensorboard

In [ ]:
x, t = torch.randn(1, 1, seq_length), torch.randn(1)
writer = SummaryWriter(OS().path_to_remote_path(f'../Results/runs/{model_name}', remote_kernel))
writer.add_graph(u_net, [x,t])
writer.close()

In [ ]:
notebook.start("--logdir=runs")

### Summarywriter

In [ ]:
summary(u_net, input_size=[(12, 1, seq_length), (12, 1)], device=device)